In [0]:
%sql
SELECT date_TRUNC('HOUR', session_start) AS session_date
, COUNT(*) AS sessions_count
, SUM(vc.session_duration)/3600.0 AS total_duration
, COUNT(DISTINCT vc.fk_commercial_id) AS commercials_count
, COUNT(DISTINCT vc.fk_tvid) AS tv_count
-- , COUNT(DISTINCT vc.prev_show_id) AS prev_show_count
-- , COUNT(DISTINCT vc.next_show_id) AS next_show_count
, COUNT(DISTINCT vc.prev_station_id) AS prev_station_count
, COUNT(DISTINCT vc.prev_vizio_epg_station) AS prev_wf_station_count
-- , COUNT(DISTINCT vc.next_station_id) AS next_station_count
-- , COUNT(DISTINCT vc.fk_location_id) AS loc_count
, COUNT(DISTINCT vc.fk_dma_id) AS dma_count
FROM prod.detection.viewing_commercials_firehose_dedup vc
JOIN prod.detection.commercial_id_external_firehose cief
  ON cief.external_id = vc.external_id
JOIN prod.detection.clients cl
  ON cl.client_id = cief.fk_client_id
WHERE vc.session_start >= CURRENT_DATE
  AND vc.fk_zoo_id = 17
  AND cl.client_name = 'kinetiq'
GROUP BY 1

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_dma_hourly;
CREATE TABLE dev.mohit_gangwani.ad_dma_hourly AS
SELECT DATE_TRUNC('HOUR', vc.session_start) AS session_hour
, vc.external_id AS ad_id
, vc.fk_dma_id
, COUNT(DISTINCT vc.fk_tvid) AS tv_count
, COUNT(DISTINCT vc.fk_tvid||'_'||vc.session_start) AS impression_count
FROM prod.detection.viewing_commercials_firehose_dedup vc
JOIN prod.detection.commercial_id_external_firehose cief
  ON cief.external_id = vc.external_id
JOIN prod.detection.clients cl
  ON cl.client_id = cief.fk_client_id
WHERE vc.session_start >= CURRENT_DATE - 7
  AND vc.session_start < CURRENT_DATE
  AND vc.fk_zoo_id = 17
  AND cl.client_name = 'kinetiq'
  AND vc.fk_dma_id IS NOT NULL
GROUP BY 1, 2, 3;

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_dma_overall;
CREATE TABLE dev.mohit_gangwani.ad_dma_overall AS
SELECT vc.external_id AS ad_id
, vc.fk_dma_id
, COUNT(DISTINCT vc.fk_tvid) AS tv_count
, COUNT(DISTINCT vc.fk_tvid||'_'||vc.session_start) AS impression_count
FROM prod.detection.viewing_commercials_firehose_dedup vc
JOIN prod.detection.commercial_id_external_firehose cief
  ON cief.external_id = vc.external_id
JOIN prod.detection.clients cl
  ON cl.client_id = cief.fk_client_id
WHERE vc.session_start >= CURRENT_DATE - 7
  AND vc.session_start < CURRENT_DATE
  AND vc.fk_zoo_id = 17
  AND cl.client_name = 'kinetiq'
  AND vc.fk_dma_id IS NOT NULL
GROUP BY 1, 2
HAVING impression_count >= 20
   AND tv_count >= 10;

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.dma_opportunities_hourly;
CREATE TABLE dev.mohit_gangwani.dma_opportunities_hourly AS
SELECT DATE_TRUNC('HOUR', vc.session_start) AS session_hour
, vc.fk_dma_id
, COUNT(DISTINCT vc.fk_tvid) AS active_tvs
FROM prod.detection.viewing_commercials_firehose_dedup vc
JOIN prod.detection.commercial_id_external_firehose cief
  ON cief.external_id = vc.external_id
JOIN prod.detection.clients cl
  ON cl.client_id = cief.fk_client_id
WHERE vc.session_start >= CURRENT_DATE - 7
  AND vc.session_start < CURRENT_DATE
  AND vc.fk_zoo_id = 17
  AND cl.client_name = 'kinetiq'
GROUP BY 1, 2;

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_dmas_count;
CREATE TABLE dev.mohit_gangwani.ad_dmas_count AS
WITH
-- ad_filter AS (
--   SELECT ad_id, impression_count, tv_count
--   FROM dev.mohit_gangwani.ad_dma_overall
--   WHERE impression_count >= 20
--     AND tv_count >= 10
--   GROUP BY ALL
-- )
-- , 
ad_dma AS (
  SELECT a.ad_id, a.fk_dma_id, a.impression_count, a.tv_count
  FROM dev.mohit_gangwani.ad_dma_overall a
  -- JOIN ad_filter f
  --   ON a.ad_id = f.ad_id
  GROUP BY ALL
)
SELECT a.ad_id
, COUNT(DISTINCT a.fk_dma_id) AS dma_count
, SUM(a.impression_count) AS ttl_impressions
, SUM(a.tv_count) AS tv_count
, dma_count*1.0/ 210 AS dma_coverage
FROM ad_dma a
GROUP BY 1;

In [0]:
%sql
SELECT * FROM dev.mohit_gangwani.ad_dmas_count
WHERE dma_count <= 2
ORDER BY ttl_impressions DESC
LIMIT 10

In [0]:
%sql
SELECT * FROM dev.mohit_gangwani.ad_dma_overall
WHERE ad_id = 'AE16546-2025-30-04769'

In [0]:
%sql
SELECT * FROM dev.mohit_gangwani.ad_dmas_count
WHERE dma_count >= 100
ORDER BY ttl_impressions DESC
LIMIT 10

In [0]:
%sql
SELECT * FROM dev.mohit_gangwani.ad_dmas_count
WHERE dma_count >= 3
  AND dma_count < 100
ORDER BY ttl_impressions DESC
LIMIT 10

In [0]:
%sql
SELECT * FROM dev.mohit_gangwani.ad_dma_overall
WHERE ad_id IN ('AE16546-2025-43-05892', 'AE16546-2024-45-01430', 'AE16546-2025-43-07032', 'AE16546-2025-43-05083', 'AE16546-2025-43-05730', 'AE16546-2025-43-04818', 'AE16546-2025-41-03971', 'AE16546-2023-42-05587', 'AE16546-2025-09-01406', 'AE16546-2024-42-01939')

In [0]:
%sql
SELECT APPROX_PERCENTILE(dma_count, 0.25) AS perc_25
, APPROX_PERCENTILE(dma_count, 0.50) AS perc_50
, APPROX_PERCENTILE(dma_count, 0.75) AS perc_75
, APPROX_PERCENTILE(dma_count, 0.90) AS perc_90
, AVG(dma_count) AS avg_dma_count
FROM dev.mohit_gangwani.ad_dmas_count

In [0]:
%sql
SELECT dma_count
, COUNT(DISTINCT ad_id)
FROM dev.mohit_gangwani.ad_dmas_count
GROUP BY 1

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT COUNT(DISTINCT ad_id)
FROM dev.mohit_gangwani.ad_dmas_count

In [0]:
import pandas as pd
import numpy as np

In [0]:
def gini(ad):
    x = np.sort(np.array(ad))
    n = x.size
    if n <= 1 or np.sum(x) == 0:
        return 1.0

    index = np.arange(1, n + 1)
    a = (2.0 * np.sum(index * x))/(n * np.sum(x))
    c = (n + 1) / n
    g = a - c
    norm = n / (n - 1)
    return norm * g

In [0]:
def locality_index(values):
    if len(values) <= 1 or np.sum(values) == 0:
        return 1.0
    p = np.array(values) / np.sum(values)
    p = p[p > 0]
    H = -np.sum(p * np.log(p))
    return 1 - (H / np.log(len(p)))

In [0]:
def get_gini_entropy_df(df, ad_ids):
    x_df = df[df.ad_id.isin(ad_ids)].copy()
    x_df.sort_values(by=['ad_id', 'impression_count'], inplace=True)
    x_df.reset_index(inplace=True, drop=True)
    ndf = pd.DataFrame()
    for i, ad_id in enumerate(ad_ids):
        a = x_df[x_df['ad_id'] == ad_id]
        ndf.loc[i, 'ad_id'] = ad_id
        ndf.loc[i, 'impression_count'] = a.impression_count.sum()
        ndf.loc[i, 'dma_count'] = a.fk_dma_id.nunique()
        ndf.loc[i, 'gini'] = gini(a['impression_count'])
        ndf.loc[i, 'entropy'] = locality_index(a['impression_count'])
    return ndf